In [ ]:
import statsmodels.api as sm
from statsmodels.stats.sandwich_covariance import cov_hac
from stargazer.stargazer import Stargazer

def stargazer_hac(models, type_out="text", omit_stats=None, out=None):
    """
    Generates regression output tables with Newey-West standard errors.

    Parameters:
    models (list): List of fitted regression models (from statsmodels).
    type_out (str): Output format ("text", "html", or "latex"). Default is "text".
    omit_stats (list): Statistics to omit from the output, e.g., ['f', 'rsquared'].
    out (str): File path to save the output, None to display.

    Returns:
    Displays or saves the regression output based on the specified format.
    """
    if not isinstance(models, list):
        models = [models]
    
    # Applying Newey-West standard errors
    for model in models:
        nw_cov = cov_hac(model)
        model.cov_params_default = nw_cov

    # Formatting the output using Stargazer
    stargazer = Stargazer(models)
    if omit_stats:
        stargazer.omit_statistics(omit_stats)
    stargazer.add_custom_notes(["Newey-West standard errors in parentheses"])

    # Generating output
    if type_out == "html":
        output = stargazer.render_html()
    elif type_out == "latex":
        output = stargazer.render_latex()
    else:
        output = stargazer.render_text()

    if out:
        with open(out, 'w') as file:
            file.write(output)
    else:
        print(output)

# Example usage
if __name__ == '__main__':
    # Assume data is loaded and appropriate models are fitted
    # Fit example models using statsmodels
    data = sm.datasets.get_rdataset("Guerry", "HistData").data
    model1 = sm.OLS.from_formula('Lottery ~ Literacy + np.log(Pop1831)', data=data).fit()
    model2 = sm.OLS.from_formula('Lottery ~ Literacy + Wealth + np.log(Pop1831)', data=data).fit()

    # Display or save output
    stargazer_hac([model1, model2], type_out='text')
